In [67]:
import torch
# We start by defining the matrix A
A = torch.tensor([[1.,2.],[3.,4.]])
# A = torch.rand(2, 2) 

In [68]:
print(A)

tensor([[1., 2.],
        [3., 4.]])


In [69]:
# We now define a function Y
# Y has inputs x (column vector) and A (a matrix)
# Y returns X^T A
def Y(x,A):
    x_transpose = torch.transpose(x,0,1)
    y = torch.matmul(x_transpose, A)
    return y

In [70]:
# We will now test the function Y
x = torch.tensor([[1.],[2.]])
y = Y(x,A)
print(y)

tensor([[23., 34.]])


In [85]:
# We will now attempt to find the Jacobian of Y

der_Y = torch.autograd.functional.jacobian(Y,(x,A))
print(der_Y)

(tensor([[[[1.],
          [3.]],

         [[2.],
          [4.]]]]), tensor([[[[0.5822, 0.0000],
          [0.4210, 0.0000]],

         [[0.0000, 0.5822],
          [0.0000, 0.4210]]]]))


In [72]:
dYdx = der_Y[0].squeeze()
print(dYdx)

tensor([[1., 3.],
        [2., 4.]])


In [73]:
# We will now test the Jacobian of Y
del_x = torch.matmul(dYdx, x)
print(del_x)

tensor([[23.],
        [34.]])


In [74]:
# We will now define a function J
# J = 1/2 (x.t Ax)
def J(x,A):
    y = Y(x,A)
    j =0.5 * torch.matmul(y,x)
    return j

In [75]:
# We will test the function
j = J(x,A)
print(j)

tensor([[159.5000]])


In [76]:
der_J = torch.autograd.functional.jacobian(J,(x,A))
print(der_J)

(tensor([[[[20.0000],
          [36.5000]]]]), tensor([[[[12.5000, 15.0000],
          [15.0000, 18.0000]]]]))


In [77]:
# del_j = der_J[0] * x
djdx = der_J[0].squeeze()
print(djdx)
print(torch.matmul(djdx,x))

tensor([20.0000, 36.5000])
tensor([319.])


In [80]:
# We will now define our perturbation
x_pert = torch.rand(2,1)*10**(-5)
print("x'",x_pert)
x = torch.rand(2,1)
print('x:', x)
der_J = torch.autograd.functional.jacobian(J,(x,A))
djdx = der_J[0].squeeze()

x' tensor([[8.9840e-07],
        [4.4166e-06]])
x: tensor([[0.5822],
        [0.4210]])


We want to see how well the TLM can emulate the small perturbation
$|J(x + x',A) - J(x,A) - \frac{\partial J}{\partial x} x'|$

In [83]:
TLM_test_result = abs(J(x+x_pert,A) - J(x,A) - torch.matmul(djdx, x_pert))
print(TLM_test_result[0])

tensor([1.6346e-07])


We want to define a function $J(x,A) = x^T A x$